# Brane in outgoing AdS$_5$–Vaidya: symbolic and numerical cross-check of the junction closure

**Status.** Independent verification notebook. It does **not** claim new physics: the energy-exchange law
$\dot\rho+3H(\rho+p)=2\,T_{AB}n^AU^B$ for a $\mathbb Z_2$ FRW brane with bulk matter is known (e.g. Apostolopoulos & Tetradis,
hep-th/0412246, Eq. 29, which they state reproduces Vaidya–AdS cosmologies). The purpose is to check, end to end and with explicit
conventions, that the brane trajectory, both Israel equations, the Codazzi projection and the outgoing-Vaidya source are mutually consistent,
and to isolate which factors (notably $\dot u^2$) a naive "$Q\propto\dot\mu/a^4$" identification misses.

**Authorship and AI-assistance statement (author to complete before publishing).**
Javier Manuel Martín Alonso ran every cell of this notebook on 2026-10-10 in JupyterLab (Python 3.14.2, numpy 2.5.3, sympy 1.14.0; OS: [Windows 10]) and is responsible for its content.
The notebook was drafted with the assistance of an AI assistant (Perplexity); the author checked [the derivation steps and the numerical tests].

**Conventions.** Signature $(-++++)$; $\kappa_5^2=8\pi G_5$; $f(u,r)=1+r^2/L^2-\mu(u)/r^2$ ($k=1$); outgoing coordinates
$ds^2=-f\,du^2-2\,du\,dr+r^2d\Omega_3^2$; brane at $r=a(\tau)$, $u=u(\tau)$; normal $n_A=(\dot a,-\dot u)$ pointing to decreasing $r$ (into the retained bulk);
Israel ($\mathbb Z_2$): $K_{\mu\nu}=-\tfrac{\kappa_5^2}{2}(S_{\mu\nu}-\tfrac13 S h_{\mu\nu})$, $S_{\mu\nu}=-\lambda h_{\mu\nu}+T_{\mu\nu}$.

**How to run.** `pip install -r requirements.txt`, then `jupyter lab` (or run all cells top to bottom). Runtime: a few seconds.

In [1]:
import numpy as np, sympy as sp
from scipy.integrate import solve_ivp
rng = np.random.default_rng(7)
print('numpy', np.__version__, '| sympy', sp.__version__)


numpy 2.5.3 | sympy 1.14.0


## 1. Einstein equations: the outgoing-Vaidya source

In [2]:
u,r,chi,th,ph = sp.symbols('u r chi theta phi', positive=True)
L = sp.symbols('L', positive=True)
mu = sp.Function('mu')
X = [u,r,chi,th,ph]
f = 1 + r**2/L**2 - mu(u)/r**2
g = sp.zeros(5,5)
g[0,0] = -f; g[0,1] = g[1,0] = -1
g[2,2] = r**2; g[3,3] = r**2*sp.sin(chi)**2; g[4,4] = r**2*sp.sin(chi)**2*sp.sin(th)**2
gi = sp.simplify(g.inv())
def Gam(a,b,c):
    return sum(gi[a,d]*(sp.diff(g[d,b],X[c])+sp.diff(g[d,c],X[b])-sp.diff(g[b,c],X[d])) for d in range(5))/2
Gm = [[[sp.simplify(Gam(a,b,c)) for c in range(5)] for b in range(5)] for a in range(5)]
def Ric(b,c):
    s = 0
    for a in range(5):
        s += sp.diff(Gm[a][b][c],X[a]) - sp.diff(Gm[a][b][a],X[c])
        for d in range(5):
            s += Gm[a][a][d]*Gm[d][b][c] - Gm[a][c][d]*Gm[d][b][a]
    return sp.simplify(s)
Rm = sp.Matrix(5,5,lambda i,j: Ric(i,j) if i<=j else 0)
for i in range(5):
    for j in range(i): Rm[i,j] = Rm[j,i]
Rs = sp.simplify(sum(gi[i,j]*Rm[i,j] for i in range(5) for j in range(5)))
E = sp.simplify(Rm - Rs*g/2 + (-6/L**2)*g)     # G_AB + Lambda g_AB, Lambda = -6/L^2
print('G_uu + Lambda g_uu =', E[0,0])
print('all other components zero:', all(sp.simplify(E[i,j])==0 for i in range(5) for j in range(5) if (i,j)!=(0,0)))
# => kappa^2 T_uu = -3 mu'(u)/(2 r^3):  T_AB = Phi k_A k_B, k_A = -d_A u,  Phi = -3 mu'/(2 kappa^2 r^3); NEC <=> mu' <= 0

G_uu + Lambda g_uu = -3*Derivative(mu(u), u)/(2*r**3)
all other components zero: True


## 2. Brane kinematics, normal, and extrinsic curvature

In [3]:
A, Ad, Add, Ud, Udd, Uu = sp.symbols('A Ad Add Ud Udd Uu')
m0, m1, ell, Lv = sp.symbols('m0 m1 ell Lv', positive=True)
mu_c = lambda uu: m0 + m1*sp.exp(-uu/ell)
fU = lambda uu, rr: 1 + rr**2/Lv**2 - mu_c(uu)/rr**2
Ur, Rr = sp.symbols('Ur Rr')
f2 = fU(Ur, Rr)
g2 = sp.Matrix([[-f2,-1],[-1,0]]); gi2 = g2.inv(); X2=[Ur,Rr]
def G2(a,b,c):
    return sum(gi2[a,d]*(sp.diff(g2[d,b],X2[c])+sp.diff(g2[d,c],X2[b])-sp.diff(g2[b,c],X2[d])) for d in range(2))/2
S = {Ur:Uu, Rr:A}
Uv = [Ud, Ad]; Uvd = [Udd, Add]
n_low = [Ad, -Ud]
acc = [Uvd[a_] + sum(G2(a_,b_,c_).subs(S)*Uv[b_]*Uv[c_] for b_ in range(2) for c_ in range(2)) for a_ in range(2)]
Ktt = -sum(n_low[a_]*acc[a_] for a_ in range(2))
fs, fr, fu = f2.subs(S), sp.diff(f2,Rr).subs(S), sp.diff(f2,Ur).subs(S)
B = fs*Ud + Ad
cons_d = fu*Ud**3 + fr*Ad*Ud**2 + 2*fs*Ud*Udd + 2*Add*Ud + 2*Ad*Udd   # d/dtau of f*Ud^2 + 2*Ad*Ud - 1
Udd_sol = sp.solve(cons_d, Udd)[0]
Ktt_s = Ktt.subs(Udd, Udd_sol)

def random_point():
    pars = {m0: rng.uniform(0.1,0.8), m1: rng.uniform(0.2,1.5), ell: rng.uniform(1,4), Lv: rng.uniform(0.8,1.6),
            A: rng.uniform(1.5,3.0), Uu: rng.uniform(0,2), Ad: rng.uniform(0.2,1.2), Add: rng.uniform(-0.5,0.5)}
    f0 = float(fs.subs(pars))
    ud0 = (-pars[Ad] + np.sqrt(pars[Ad]**2 + f0))/f0       # outgoing branch of the normalisation
    pars[Ud] = ud0
    return pars

errs=[]
for _ in range(20):
    P = random_point()
    lhs = float(Ktt_s.subs(P)); rhs = float(((Add + fr/2 + fu*Ud**2/2)/B).subs(P))
    errs.append(abs(lhs-rhs))
print('max |K_tautau(from Christoffels) - (a_ddot + f_r/2 + f_u udot^2/2)/B| =', max(errs))

n_u, n_r = sp.symbols('n_u n_r')
Kchichi = -(Gm[0][2][2]*n_u + Gm[1][2][2]*n_r)               # K_ij = -Gamma^A_ij n_A (full 5D Christoffels from Part 1)
chk = sp.simplify(Kchichi.subs({n_u:sp.Symbol('Ad'), n_r:-sp.Symbol('Ud')}))
expected = -r*(sp.Symbol('Ad') + f*sp.Symbol('Ud'))          # -(B/a) g_chichi with B = f udot + adot, r = a
print('K_chichi = -(B/a) g_chichi :', sp.simplify(chk - expected) == 0)


max |K_tautau(from Christoffels) - (a_ddot + f_r/2 + f_u udot^2/2)/B| = 7.771561172376096e-16
K_chichi = -(B/a) g_chichi : True


## 3. Closure of the Israel equations: derivation of the exchange law

In [4]:
rho, p, rhod, kap, lam, a_, ad_, fu_, ud_ = sp.symbols('rho p rhod kappa lambda a ad fu ud')
# (i) spatial Israel: B = kappa^2 (rho+lam) a/6 ; (ii) temporal Israel: K_tt = -kappa^2 (2 rho + 3 p - lam)/6
# geometric identity (checked numerically below): Bdot = adot*K_tt + (1/2) f_u udot^2
Ktt_isr = -kap**2*(2*rho+3*p-lam)/6
eq = sp.Eq(kap**2*(rhod*a_ + (rho+lam)*ad_)/6, ad_*Ktt_isr + fu_*ud_**2/2)
rhod_sol = sp.simplify(sp.solve(eq, rhod)[0])
H = ad_/a_
claimed = -3*H*(rho+p) + 3*fu_*ud_**2/(kap**2*a_)
print('rho_dot =', rhod_sol)
print('equals  -3H(rho+p) + 3 f_u udot^2/(kappa^2 a):', sp.simplify(rhod_sol-claimed)==0)

errs=[]
for _ in range(20):
    P = random_point()
    Bdot = (Ad*Add + (fr*Ad + fu*Ud)/2)/B          # d/dtau sqrt(adot^2+f)
    lhs = float(Bdot.subs(P)); rhs = float((Ad*Ktt_s + fu*Ud**2/2).subs(P))
    errs.append(abs(lhs-rhs))
print('max |Bdot - (adot K_tt + f_u udot^2/2)| =', max(errs))
# f_u = -mu'(u)/a^2  =>  Q = -3 mu'(u) udot^2 /(kappa^2 a^3)

rho_dot = 3*(-ad*kappa**2*(p + rho) + fu*ud**2)/(a*kappa**2)
equals  -3H(rho+p) + 3 f_u udot^2/(kappa^2 a): True
max |Bdot - (adot K_tt + f_u udot^2/2)| = 5.551115123125783e-16


## 4. Codazzi projection with the Vaidya source gives the same current

In [5]:
Phi = lambda mup, aa, k2: -3*mup/(2*k2*aa**3)            # T_AB = Phi k_A k_B
# k_A = (-1,0): k.n = -n^u = -udot ; k.U = -udot  =>  T_AB n^A U^B = Phi udot^2 ;  Q = -u_mu Q^mu = 2 T_AB n^A U^B
Q_codazzi = lambda mup, aa, ud, k2: 2*Phi(mup,aa,k2)*ud**2
Q_israel  = lambda mup, aa, ud, k2: 3*(-mup/aa**2)*ud**2/(k2*aa)
x1 = Q_codazzi(-0.37, 2.2, 0.31, 1.0); x2 = Q_israel(-0.37, 2.2, 0.31, 1.0)
print('Codazzi:', x1, ' Israel:', x2, ' equal:', np.isclose(x1,x2))

Codazzi: 0.010017937640871522  Israel: 0.010017937640871522  equal: True


## 5. Dynamical consistency test (numerical)

Evolve $(a,u,\rho)$ with the spatial Israel equation defining $\dot a$, the normalisation defining $\dot u=1/(B+\dot a)$, and $\dot\rho=-3H(1+w)\rho+Q$.
Then test the **independent** acceleration (temporal Israel) equation. A wrong $Q$ leaves a non-zero residual.
RS tuning: $\kappa_5^2\lambda/6=1/L$ (here $\kappa_5=L=1$, $\lambda=6$).

In [6]:
kap5, Lc, lam5, w = 1.0, 1.0, 6.0, 1/3
m0v, m1v, ellv = 0.2, 1.5, 3.0
mu_f  = lambda uu: m0v + m1v*np.exp(-uu/ellv)
mup_f = lambda uu: -m1v/ellv*np.exp(-uu/ellv)
f_f   = lambda uu, aa: 1 + aa**2/Lc**2 - mu_f(uu)/aa**2
fr_f  = lambda uu, aa: 2*aa/Lc**2 + 2*mu_f(uu)/aa**3
fu_f  = lambda uu, aa: -mup_f(uu)/aa**2
def kin(uu, aa, rr):
    Bv = kap5**2*(rr+lam5)*aa/6
    adv = np.sqrt(max(Bv**2 - f_f(uu,aa), 0.0))
    udv = 1.0/(Bv + adv)
    return Bv, adv, udv
def Qmode(mode, uu, aa, udv):
    Qtrue = -3*mup_f(uu)*udv**2/(kap5**2*aa**3)
    return {'correct':Qtrue, 'no_udot_sq': -3*mup_f(uu)/(kap5**2*aa**3), 'half': 0.5*Qtrue}[mode]
def make_rhs(mode):
    def rhs(t, y):
        aa, uu, rr = y
        Bv, adv, udv = kin(uu, aa, rr)
        return [adv, udv, -3*(adv/aa)*(1+w)*rr + Qmode(mode, uu, aa, udv)]
    return rhs
def residual(sol, mode):
    res=[]; scale=[]
    for i in range(sol.t.size):
        aa, uu, rr = sol.y[:,i]
        Bv, adv, udv = kin(uu, aa, rr)
        rd = -3*(adv/aa)*(1+w)*rr + Qmode(mode, uu, aa, udv)
        Bd = kap5**2*(rd*aa + (rr+lam5)*adv)/6
        fd = fr_f(uu,aa)*adv + fu_f(uu,aa)*udv
        add_pred = (Bv*Bd - 0.5*fd)/adv                              # d/dtau sqrt(B^2 - f)
        add_isr  = -Bv*kap5**2*(2*rr+3*w*rr-lam5)/6 - 0.5*fr_f(uu,aa) - 0.5*fu_f(uu,aa)*udv**2
        res.append(add_pred - add_isr); scale.append(abs(add_pred)+1e-30)
    return np.max(np.abs(res)), np.max(np.abs(res)/np.array(scale))
y0 = [3.0, 0.0, 2.0]
for mode in ['correct','no_udot_sq','half']:
    sol = solve_ivp(make_rhs(mode), [0,3.0], y0, rtol=1e-11, atol=1e-13, t_eval=np.linspace(0,3,60))
    print(f'{mode:12s} max |residual| = %.2e   max relative = %.2e' % residual(sol, mode))

correct      max |residual| = 2.36e-15   max relative = 1.03e-14
no_udot_sq   max |residual| = 4.81e-02   max relative = 2.70e-01
half         max |residual| = 5.64e-04   max relative = 2.64e-03


## 6. Dark-radiation interface

With $\rho_{DR}=3\mu/(\kappa_4^2a^4)$, $\kappa_4^2=\kappa_5^4\lambda/6$ (RS tuning): $\dot\rho_{DR}+4H\rho_{DR}=3\mu'(u)\dot u/(\kappa_4^2a^4)$ — one power of $\dot u$, versus $\dot u^2$ in $Q$.

In [7]:
tau_s = sp.symbols('tau_s'); k4 = sp.symbols('kappa4sq', positive=True)
a_f = sp.Function('a_f')(tau_s); u_f = sp.Function('u_f')(tau_s); mu_s = sp.Function('mu_s')
rhoDR = 3*mu_s(u_f)/(k4*a_f**4)
lhs_s = sp.diff(rhoDR, tau_s) + 4*sp.diff(a_f,tau_s)/a_f*rhoDR
rhs_s = 3*sp.diff(mu_s(u_f), tau_s)/(k4*a_f**4)
print('symbolic identity holds:', sp.simplify(lhs_s - rhs_s) == 0)

kap4sq = kap5**4*lam5/6
sol = solve_ivp(make_rhs('correct'), [0,3.0], y0, rtol=1e-12, atol=1e-14, method='DOP853', t_eval=np.linspace(0,3,6001))
aa, uu, rr = sol.y
kk = np.array([kin(uu[i], aa[i], rr[i]) for i in range(aa.size)])
Bv, adv, udot = kk[:,0], kk[:,1], kk[:,2]
Hh = adv/aa
rho_DR = 3*mu_f(uu)/(kap4sq*aa**4)
d_rho = np.gradient(rho_DR, sol.t, edge_order=2)
analytic = 3*mup_f(uu)*udot/(kap4sq*aa**4) - 12*mu_f(uu)*adv/(kap4sq*aa**5)
print('d rho_DR/dtau: finite differences vs chain rule (rel. to max) =', np.max(np.abs(d_rho-analytic))/np.max(np.abs(analytic)))

# exact RS-tuned form: B = a(1+rho/lambda)/L  =>  udot = 1/(B+adot) = L/(a(1+rho/lambda+H L))
udot_rs = Lc/(aa*(1 + rr/lam5 + Hh*Lc))
print('max rel. diff udot vs RS-tuned closed form =', np.max(np.abs(udot-udot_rs)/udot))
print("=> Q = -3 mu'(u) L^2 / (kappa^2 a^5 (1+rho/lambda+H L)^2)   [exact for kappa^2 lambda/6 = 1/L]")

symbolic identity holds: True
d rho_DR/dtau: finite differences vs chain rule (rel. to max) = 3.3569920826155745e-06
max rel. diff udot vs RS-tuned closed form = 3.957594747213407e-16
=> Q = -3 mu'(u) L^2 / (kappa^2 a^5 (1+rho/lambda+H L)^2)   [exact for kappa^2 lambda/6 = 1/L]


## 7. What this notebook does and does not establish

**Established (under the stated assumptions):** consistency of $K_{ij}$, $K_{\tau\tau}$, both Israel equations, the Codazzi projection and the outgoing-Vaidya source; the NEC condition $\partial_u\mu\le0$; failure of two naive alternatives in the numerical test.

**Not established / open:** (i) novelty relative to the braneworld literature with bulk radiation (Binétruy et al.; Langlois–Sorbo–Rodríguez-Martínez, PRL 89 171301; Apostolopoulos–Tetradis, hep-th/0412246; Tetradis, CQG 21 5221; Jennings et al.; Kraus; Ida); (ii) the physical origin of $\mu(u)$: an outgoing positive-energy flux that extends to the horizon is a white-hole-like (time-reversed collapse) configuration; a black-hole evaporation model needs ingoing negative-energy flux near the horizon (Hiscock-type patching) and an outgoing flux only far from it; (iii) asymmetric (non-$\mathbb Z_2$) branes; (iv) stability of the coupled bulk–brane system.